# LinkedIn Visuals: The Hidden Structure of BERT

**Stunning dark-theme visualizations of neural network compressibility via SVD**

This notebook generates publication-ready images for a LinkedIn post about structural compression of transformer models. All visualizations are computed directly from BERT's weight matrices using Singular Value Decomposition — no pre-saved results needed.

**Output:** 3 high-resolution images saved to `results/` + LinkedIn post text.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patheffects as pe
from transformers import AutoModel
import os, sys, warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════
#  DARK THEME + NEON PALETTE
# ═══════════════════════════════════════════════════════

DARK_BG      = '#0d1117'
DARK_GRID    = '#21262d'
TEXT_PRIMARY  = '#e6edf3'
TEXT_SECONDARY = '#8b949e'

COMP_NAMES  = ['Query', 'Key', 'Value', 'Attn Out', 'FFN Inter', 'FFN Out']
COMP_COLORS = ['#ff006e', '#ff9500', '#ffd60a', '#00ff87', '#00d4ff', '#7b2ff7']
COMP_COLOR_MAP = dict(zip(COMP_NAMES, COMP_COLORS))

NEON_CMAP = LinearSegmentedColormap.from_list(
    'neon', ['#00ff87', '#00d4ff', '#7b2ff7', '#ff006e'], N=256)

plt.rcParams.update({
    'figure.facecolor': DARK_BG,
    'axes.facecolor':   DARK_BG,
    'text.color':       TEXT_PRIMARY,
    'axes.labelcolor':  TEXT_PRIMARY,
    'xtick.color':      TEXT_SECONDARY,
    'ytick.color':      TEXT_SECONDARY,
    'axes.edgecolor':   DARK_GRID,
    'grid.color':       DARK_GRID,
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.facecolor': DARK_BG,
    'savefig.bbox':     'tight',
    'font.size':        12,
})

print('Dark theme configured')
print('Neon palette ready')

In [ ]:
# ═══════════════════════════════════════════════════════
#  LOAD MODEL + COMPUTE SVD FOR ALL 72 WEIGHT MATRICES
# ═══════════════════════════════════════════════════════

project_root = os.path.dirname(os.path.abspath(''))
results_dir  = os.path.join(project_root, 'results')
os.makedirs(results_dir, exist_ok=True)

# Try fine-tuned model first, fall back to pretrained
model_path = os.path.join(project_root, 'results', 'bert-goemotions-final')
try:
    from transformers import AutoModelForSequenceClassification
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    MODEL_LABEL = 'BERT fine-tuned on GoEmotions'
    print(f'Loaded fine-tuned model from {model_path}')
except Exception:
    model = AutoModel.from_pretrained('bert-base-uncased')
    MODEL_LABEL = 'BERT-base-uncased (pretrained)'
    print(f'Fine-tuned model not found — using pretrained BERT-base-uncased')

model.eval()
print(f'Model: {MODEL_LABEL}')

# ── Component matching (order matters: more specific patterns first) ──
comp_patterns = [
    ('attention.self.query',  'Query'),
    ('attention.self.key',    'Key'),
    ('attention.self.value',  'Value'),
    ('attention.output.dense','Attn Out'),
    ('intermediate.dense',    'FFN Inter'),
]

spectral = {}

for name, param in model.named_parameters():
    if 'encoder.layer' not in name or 'weight' not in name or 'LayerNorm' in name:
        continue

    # Extract layer index
    parts = name.split('.')
    layer_idx = None
    for k, p in enumerate(parts):
        if p == 'layer':
            layer_idx = int(parts[k + 1])
            break
    if layer_idx is None:
        continue

    # Match component type
    comp = None
    for pattern, cname in comp_patterns:
        if pattern in name:
            comp = cname
            break
    if comp is None:
        if 'output.dense' in name:
            comp = 'FFN Out'
        else:
            continue

    # SVD decomposition
    W = param.data.float()
    _, S, _ = torch.linalg.svd(W, full_matrices=False)
    S = S.cpu().numpy()

    total_e = np.sum(S ** 2)
    cum_e   = np.cumsum(S ** 2) / total_e

    ranks = {}
    for t in [0.80, 0.85, 0.90, 0.95, 0.99]:
        ranks[t] = int(np.searchsorted(cum_e, t) + 1)

    spectral[(layer_idx, comp)] = {
        'S': S, 'cum_energy': cum_e, 'ranks': ranks, 'shape': tuple(W.shape)
    }

print(f'\nSVD computed for {len(spectral)} weight matrices')
print(f'Architecture: 12 encoder layers x 6 components = 72 matrices\n')
for comp in COMP_NAMES:
    avg = np.mean([spectral[(l, comp)]['ranks'][0.95] for l in range(12)])
    print(f'  {comp:12s}  avg rank @ 95% energy = {avg:.0f}')

## 1. The Compressibility Fingerprint

A **radial polar heatmap** of BERT's 72 weight matrices — like an MRI scan of a neural network.

- **12 angular sectors** = encoder layers (L0 – L11)
- **6 concentric rings** = component types (Query, Key, Value, Attn Out, FFN Inter, FFN Out)
- **Color** = minimum rank needed for 95% energy retention (green = compressible, pink = hard)

In [ ]:
# ═══════════════════════════════════════════════════════
#  VIZ 1 — THE COMPRESSIBILITY FINGERPRINT (HERO IMAGE)
# ═══════════════════════════════════════════════════════

fig = plt.figure(figsize=(16, 16))
ax  = fig.add_subplot(111, projection='polar')

n_layers, n_comps = 12, 6

# Data: rank needed for 95% energy
rank_data = np.array([
    [spectral[(j, c)]['ranks'][0.95] for j in range(n_layers)]
    for c in COMP_NAMES
])

# Polar coordinates
theta   = np.linspace(0, 2 * np.pi, n_layers, endpoint=False)
bar_w   = 2 * np.pi / n_layers * 0.82
ring_h  = 1.0
gap     = 0.15
r_bases = np.array([2.0 + i * (ring_h + gap) for i in range(n_comps)])

vmin, vmax = rank_data.min(), rank_data.max()
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

# ── Draw cells ──
for i in range(n_comps):
    for j in range(n_layers):
        color = NEON_CMAP(norm(rank_data[i, j]))
        ax.bar(theta[j], ring_h, width=bar_w, bottom=r_bases[i],
               color=color, edgecolor=DARK_BG, linewidth=0.8, alpha=0.92)

# ── Component labels (left side) ──
for i, comp in enumerate(COMP_NAMES):
    ax.text(np.pi + 0.12, r_bases[i] + ring_h / 2, comp,
            fontsize=12, fontweight='bold', color=COMP_COLORS[i],
            va='center', ha='left',
            path_effects=[pe.withStroke(linewidth=4, foreground=DARK_BG)])

# ── Polar styling ──
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.set_xticks(theta)
ax.set_xticklabels([f'L{i}' for i in range(12)], fontsize=14, fontweight='bold')
ax.set_ylim(0, r_bases[-1] + ring_h + 1.5)
ax.set_yticks([])
ax.grid(True, alpha=0.06, color='#ffffff')
ax.spines['polar'].set_visible(False)

# ── Center text ──
ax.text(0, 0.0, 'BERT', fontsize=32, fontweight='bold', color=TEXT_PRIMARY,
        ha='center', va='center',
        path_effects=[pe.withStroke(linewidth=6, foreground=DARK_BG)])
ax.text(0, 0.85, '110M params', fontsize=12, color=TEXT_SECONDARY,
        ha='center', va='center',
        path_effects=[pe.withStroke(linewidth=4, foreground=DARK_BG)])

# ── Title ──
fig.suptitle('The Compressibility Fingerprint of BERT',
             fontsize=28, fontweight='bold', color=TEXT_PRIMARY, y=0.97,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])
fig.text(0.5, 0.935,
         'Minimum rank for 95% singular value energy retention  ·  72 weight matrices',
         fontsize=13, color=TEXT_SECONDARY, ha='center')

# ── Colorbar ──
sm = mpl.cm.ScalarMappable(cmap=NEON_CMAP, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.88, 0.25, 0.025, 0.3])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Minimum Rank', fontsize=12, color=TEXT_PRIMARY, labelpad=10)
cbar.ax.tick_params(colors=TEXT_SECONDARY, labelsize=10)
cbar.outline.set_edgecolor(DARK_GRID)

# ── Bottom legend ──
legend_items = [
    ('#00ff87', 'Highly compressible'),
    ('#7b2ff7', 'Moderate'),
    ('#ff006e', 'Hard to compress'),
]
for idx, (c, l) in enumerate(legend_items):
    x = 0.30 + idx * 0.18
    fig.text(x, 0.04, '\u25CF', fontsize=18, color=c, ha='center', va='center')
    fig.text(x + 0.018, 0.04, f' {l}', fontsize=12, color=TEXT_SECONDARY,
             ha='left', va='center')

plt.savefig(os.path.join(results_dir, 'linkedin_01_neural_fingerprint.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()
print(f'Saved: results/linkedin_01_neural_fingerprint.png')

## 2. The Spectral Landscape

A **ridge / joy plot** showing how singular values decay for each component type of BERT.

Each curve is the **mean across 12 layers** with a std band. The steeper the drop, the more compressible the component — its information is concentrated in fewer dimensions.

In [ ]:
# ═══════════════════════════════════════════════════════
#  VIZ 2 — THE SPECTRAL LANDSCAPE (RIDGE / JOY PLOT)
# ═══════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(18, 10))

for idx, comp in enumerate(reversed(COMP_NAMES)):
    # Gather normalized singular values from all 12 layers
    all_S = []
    for layer in range(12):
        S = spectral[(layer, comp)]['S']
        all_S.append(S / S[0])

    min_len = min(len(s) for s in all_S)
    all_S   = np.array([s[:min_len] for s in all_S])

    mean_S = np.mean(all_S, axis=0)
    std_S  = np.std(all_S, axis=0)

    x      = np.arange(min_len)
    offset = idx * 0.28
    scale  = 0.24
    color  = COMP_COLOR_MAP[comp]

    # Filled area under curve
    ax.fill_between(x, offset, offset + mean_S * scale,
                    alpha=0.55, color=color, linewidth=0, zorder=6 - idx)

    # Mean line with glow effect
    ax.plot(x, offset + mean_S * scale, color=color, linewidth=2, alpha=0.95,
            zorder=7 - idx,
            path_effects=[pe.withStroke(linewidth=5, foreground=color + '30')])

    # Std band
    upper = offset + np.clip(mean_S + std_S, 0, None) * scale
    lower = offset + np.clip(mean_S - std_S, 0, None) * scale
    ax.fill_between(x, lower, upper, alpha=0.1, color=color,
                    linewidth=0, zorder=5 - idx)

    # Component label
    ax.text(-12, offset + 0.07, comp, fontsize=14, fontweight='bold', color=color,
            ha='right', va='bottom',
            path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

# ── Reference lines at key ranks ──
for rank, label in [(64, 'r = 64'), (128, 'r = 128'), (256, 'r = 256')]:
    ax.axvline(rank, color=TEXT_SECONDARY, alpha=0.25, linestyle='--', linewidth=1)
    ax.text(rank, 6 * 0.28 + 0.08, label, fontsize=10, color=TEXT_SECONDARY,
            ha='center', va='bottom', style='italic')

# ── Annotation arrow ──
ax.annotate('Most information\nconcentrated here',
            xy=(40, 0.02), xytext=(300, 0.15),
            fontsize=12, color=TEXT_SECONDARY, style='italic',
            ha='center', va='center',
            arrowprops=dict(arrowstyle='->', color=TEXT_SECONDARY,
                            lw=1.5, connectionstyle='arc3,rad=0.2'),
            path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

# ── Styling ──
ax.set_xlim(-55, 520)
ax.set_ylim(-0.04, 6 * 0.28 + 0.18)
ax.set_xlabel('Singular Value Index', fontsize=14, labelpad=10)
ax.set_yticks([])
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)
ax.grid(True, axis='x', alpha=0.08, color='#ffffff')

ax.set_title('The Spectral Landscape of BERT\n'
             'How information concentrates in each component type',
             fontsize=22, fontweight='bold', pad=20,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'linkedin_02_spectral_landscape.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.3)
plt.show()
print(f'Saved: results/linkedin_02_spectral_landscape.png')

## 3. The Compression Hierarchy

**Not all parameters are created equal.** This chart shows how many singular values each component type needs to retain 90%, 95%, and 99% of its information — with individual layer data points scattered on top.

The punchline: Query and Key matrices can be compressed to rank ~50, while FFN Intermediate needs ~400+.

In [ ]:
# ═══════════════════════════════════════════════════════
#  VIZ 3 — THE COMPRESSION HIERARCHY
# ═══════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(15, 8))

thresholds  = [0.90, 0.95, 0.99]
thr_labels  = ['90% energy', '95% energy', '99% energy']
thr_colors  = ['#00ff87', '#00d4ff', '#ff006e']

# Compute stats per component
bar_data = {}
for comp in COMP_NAMES:
    bar_data[comp] = {}
    for t in thresholds:
        ranks = [spectral[(l, comp)]['ranks'][t] for l in range(12)]
        bar_data[comp][t] = {
            'mean': np.mean(ranks), 'std': np.std(ranks), 'all': ranks
        }

# Sort by 95% threshold (most compressible first)
sorted_comps = sorted(COMP_NAMES, key=lambda c: bar_data[c][0.95]['mean'])

y_pos   = np.arange(len(sorted_comps))
bh      = 0.25
offsets = [-bh, 0, bh]
rng     = np.random.default_rng(42)

for i, (t, tl, tc) in enumerate(zip(thresholds, thr_labels, thr_colors)):
    means = [bar_data[c][t]['mean'] for c in sorted_comps]
    stds  = [bar_data[c][t]['std']  for c in sorted_comps]

    # Bars
    ax.barh(y_pos + offsets[i], means, height=bh * 0.88,
            color=tc, alpha=0.72, edgecolor=tc, linewidth=0.5,
            label=tl, zorder=3)

    # Error bars
    ax.errorbar(means, y_pos + offsets[i], xerr=stds, fmt='none',
                ecolor=tc, alpha=0.35, capsize=3, capthick=1, zorder=4)

    # Scatter individual layers
    for j, comp in enumerate(sorted_comps):
        all_r  = bar_data[comp][t]['all']
        jitter = rng.uniform(-0.04, 0.04, 12)
        ax.scatter(all_r, np.full(12, y_pos[j] + offsets[i]) + jitter,
                   color=tc, alpha=0.30, s=18, zorder=5, edgecolors='none')

    # Value labels
    for j, (m, s) in enumerate(zip(means, stds)):
        ax.text(m + s + 12, y_pos[j] + offsets[i], f'{m:.0f}',
                fontsize=11, color=tc, va='center', fontweight='bold')

# ── Y-axis labels with component colors ──
ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_comps, fontsize=14, fontweight='bold')
for j, comp in enumerate(sorted_comps):
    ax.get_yticklabels()[j].set_color(COMP_COLOR_MAP[comp])

# ── Full-rank reference ──
max_rank = max(spectral[(0, c)]['S'].shape[0] for c in COMP_NAMES)
ax.axvline(max_rank, color=TEXT_SECONDARY, alpha=0.15, linestyle=':', linewidth=1)
ax.text(max_rank - 10, len(sorted_comps) - 0.5, f'Full rank ({max_rank})',
        fontsize=10, color=TEXT_SECONDARY, ha='right', va='bottom', style='italic')

# ── Styling ──
ax.set_xlabel('Minimum Rank Required', fontsize=14, labelpad=10)
ax.set_xlim(0, max_rank + 50)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='x', alpha=0.08, color='#ffffff')

ax.legend(fontsize=12, loc='lower right', framealpha=0.15,
          edgecolor=DARK_GRID, labelcolor=TEXT_PRIMARY,
          fancybox=True)

ax.set_title('The Compression Hierarchy\n'
             'Not all parameters are created equal',
             fontsize=22, fontweight='bold', pad=20,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'linkedin_03_compression_hierarchy.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.3)
plt.show()
print(f'Saved: results/linkedin_03_compression_hierarchy.png')

---

## LinkedIn Post

---

**Le hicimos una radiografia a un cerebro artificial. Lo que encontramos cambia la forma en que pensamos sobre la eficiencia en IA.**

Tomamos BERT — un modelo de lenguaje con 110 millones de parametros — y analizamos cada una de sus 72 matrices de pesos usando Descomposicion en Valores Singulares (SVD).

El resultado es fascinante:

No todos los parametros de una red neuronal son iguales.

Algunas matrices (como Query y Key en la atencion) concentran el 95% de su informacion en apenas ~50 de sus 768 dimensiones. Otras (como las capas FFN intermedias) la distribuyen a lo largo de 400+.

Es como si el 90% del "cerebro" fuera materia de relleno.

Esto significa que podemos comprimir un modelo de 110M parametros eliminando las dimensiones redundantes — sin perder practicamente rendimiento.

3 hallazgos clave de nuestra investigacion:

1 - La compresibilidad NO es uniforme. Cada tipo de componente tiene un "perfil espectral" unico (imagen 2). Las matrices de Query y Key caen como un acantilado — su informacion esta ultra-concentrada. Las FFN son lo opuesto.

2 - No se puede comprimir todo igual. Aplicar un rank uniforme desperdicia capacidad o destruye informacion. La compresion adaptativa — ajustando el rank por componente — es dramaticamente superior.

3 - Las capas profundas son mas sensibles. Las capas 8-11 (las mas cercanas a la tarea) pierden mas rendimiento bajo compresion que las capas tempranas. El modelo "protege" lo que mas necesita.

Conclusion: Antes de entrenar modelos mas grandes, deberiamos entender mejor la estructura de los que ya tenemos. La mayoria de los parametros no son igualmente importantes — y la SVD nos da el mapa para saber cuales si lo son.

Imagenes: huella neuronal de compresibilidad, paisaje espectral, y jerarquia de compresion de BERT.

#MachineLearning #NLP #DeepLearning #AIEfficiency #ModelCompression #BERT #SVD #TransformerModels #AIResearch

---

**Instrucciones de publicacion:**
- Sube las 3 imagenes de `results/` como carrusel o post con multiples imagenes
- La imagen 1 (fingerprint radial) es la que para el scroll — ponla primera
- La imagen 2 (spectral landscape) explica el "por que"
- La imagen 3 (compression hierarchy) da el punchline con datos concretos